![Cabecera](../../assets/cabecera_rag.png)

# Workout 1 - Crear una base de datos vectorial con ChromaDB

## Objetivos

Vas a construir el **primer tramo del motor de búsqueda semántica**:

1. Cargar documentos del corpus (agenda cultural de Madrid).
2. Trocearlos en **chunks** y convertirlos en **embeddings** (vectores con Gemini).
3. Guardar esos vectores en **ChromaDB**, una base de datos vectorial.

Al terminar tendrás una carpeta `../output/chroma_db/` que los Workouts 2 y 3 de este Sprint usarán para buscar información.

> **Nota sobre Sprint 8:** ya estudiaste ingesta, chunking y embeddings en detalle. Aquí hacemos un **recap corto** para tener datos reales y dedicar el foco a **ChromaDB**.

**Requisitos previos:**
- Tener `GEMINI_API_KEY` (Google AI Studio).
- Corpus en `02_Workout/data/`.

## Setup — Instalar librerías

Antes de programar, instalamos las dependencias del workout:

- **chromadb** — base de datos vectorial.
- **google-genai** — API de Gemini para generar embeddings.
- **langchain-community / langchain-text-splitters** — cargar PDF/TXT/MD y trocear texto (recap de S8).
- **pandas / pypdf** — leer CSV de eventos y PDF de documentación.

Dependencias a instalar:

In [ ]:
%pip install -q chromadb google-genai python-dotenv langchain-community langchain-text-splitters langchain-core pypdf pandas

## Rutas y parámetros del workout

Definimos **dónde están los datos** y **dónde guardaremos la salida**:

| Variable | Significado |
|----------|-------------|
| `DATA_DIR` | Carpeta `../data/` con FAQ, guía, PDF y CSV con datos sobre la agenda cultural de Madrid |
| `OUTPUT_DIR` | Carpeta `../output/` compartida por los 3 workouts |
| `EMBEDDINGS_JSON` | Archivo intermedio con texto + vectores |
| `CHROMA_DIR` | Carpeta donde Chroma persiste el índice |

También fijamos hiperparámetros:
- **`CHUNK_SIZE = 800`** — tamaño de cada fragmento de texto.
- **`MAX_CHUNKS_EMBED = 20`** — límite de demo (pon `None` para embeddear todo el corpus).
- **`COLLECTION_NAME`** — nombre lógico del índice dentro de Chroma.

In [ ]:
import json
import os
import re
from getpass import getpass
from pathlib import Path

import chromadb
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

WORKOUT_DIR = Path("..").resolve()
DATA_DIR = WORKOUT_DIR / "data"
OUTPUT_DIR = WORKOUT_DIR / "output"
EMBEDDINGS_JSON = OUTPUT_DIR / "embeddings.json"
CHROMA_DIR = OUTPUT_DIR / "chroma_db"

CHUNK_SIZE, CHUNK_OVERLAP = 800, 100
EMBEDDING_MODEL = "gemini-embedding-2"
MAX_CHUNKS_EMBED = 20  # None = corpus completo
COLLECTION_NAME = "agenda_cultural_madrid"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", DATA_DIR)
print("Archivos:", [p.name for p in sorted(DATA_DIR.iterdir()) if p.is_file()])

Archivos: ['206974-3-agenda-eventos-culturales-100.pdf', '206974-4-agenda-eventos-culturales-100-csv.csv', 'faq_agenda_cultural.md', 'guia_agenda_cultural.txt', 'README.md']


## Autenticación con Gemini

Los **embeddings** los genera la API de Gemini. Necesitas una clave en `GEMINI_API_KEY`.

- Si tienes un archivo `.env` en `02_Workout/`, se carga automáticamente.
- Si no, la celda te la pedirá por consola (el texto no se verá al escribir).

Sin esta clave no podrás convertir texto en vectores.

In [6]:
if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass(
        "Pega tu GEMINI_API_KEY (input oculto): "
    )

EMBEDDING_MODEL = "gemini-embedding-2"
client = genai.Client()
print("Cliente OK, modelo:", EMBEDDING_MODEL)

Cliente OK, modelo: gemini-embedding-2


## Recap Sprint 8 — Carga, limpieza y chunking

Un sistema RAG no busca en archivos enteros: busca en **fragmentos pequeños** (chunks).

En esta celda repetimos el pipeline de Sprint 8:

1. **Cargar** cada archivo de `data/` como un objeto `Document` (texto + metadata).
2. **Limpiar** espacios y saltos de línea raros.
3. **Trocear** con `RecursiveCharacterTextSplitter` (tamaño 800, solape 100).

El CSV es especial: **cada fila es un evento cultural** que convertimos a texto legible con `fila_a_texto()`.

Al final verás cuántos documentos y cuántos chunks se generaron.

In [7]:
# --- Funciones auxiliares (recap Sprint 8) ---

def valor_celda(fila, columna):
    """Lee una celda del CSV; devuelve None si está vacía."""
    if columna not in fila or pd.isna(fila[columna]):
        return None
    t = str(fila[columna]).strip()
    return t if t else None


def fila_a_texto(fila):
    """Convierte UNA fila del CSV en un párrafo legible (un evento = un texto)."""
    titulo = valor_celda(fila, "TITULO")
    if titulo is None:
        return None

    lineas = [f"Evento: {titulo}"]
    for col, etiqueta in [
        ("DESCRIPCION", "Descripción"),
        ("TITULO-ACTIVIDAD", "Actividad"),
        ("NOMBRE-INSTALACION", "Lugar"),
        ("DISTRITO-INSTALACION", "Distrito"),
        ("FECHA", "Fecha"),
        ("HORA", "Hora"),
    ]:
        v = valor_celda(fila, col)
        if v:
            lineas.append(f"{etiqueta}: {v}")

    if valor_celda(fila, "GRATUITO") == "1":
        lineas.append("Gratuito: sí")
    else:
        lineas.append("Gratuito: no")

    return "\n".join(lineas)


def cargar_corpus(data_dir: Path) -> list[Document]:
    """Lee todos los archivos de data/ y devuelve objetos Document."""
    docs: list[Document] = []

    for ruta in sorted(data_dir.iterdir()):
        if not ruta.is_file():
            continue

        antes = len(docs)
        suf = ruta.suffix.lower()

        if suf in {".txt", ".md"}:
            docs.extend(TextLoader(str(ruta), encoding="utf-8").load())
        elif suf == ".pdf":
            docs.extend(PyPDFLoader(str(ruta)).load())
        elif suf == ".csv":
            df = pd.read_csv(ruta, sep=";", encoding="latin-1")
            for _, fila in df.iterrows():
                texto = fila_a_texto(fila)
                if texto:
                    meta = {
                        "source": str(ruta),
                        "tipo": "agenda_evento",
                        "distrito": valor_celda(fila, "DISTRITO-INSTALACION"),
                    }
                    docs.append(Document(page_content=texto, metadata=meta))

        print(f"  {ruta.name}: +{len(docs) - antes} documento(s)")

    return docs


def normalizar_texto(texto: str) -> str:
    """Quita espacios de más y saltos de línea raros antes del chunking."""
    if not texto:
        return ""

    t = texto.replace("\r\n", "\n").replace("\r", "\n")
    t = re.sub(r"\n{3,}", "\n\n", t)   # no más de 2 saltos seguidos
    t = re.sub(r"[ \t]+", " ", t)       # espacios múltiples → uno solo
    return "\n".join(linea.strip() for linea in t.split("\n")).strip()


# --- Ejecución del pipeline: carga → limpieza → chunking ---

# Paso 1: cargar documentos crudos desde data/
crudos = cargar_corpus(DATA_DIR)

# Paso 2: limpiar texto y descartar documentos vacíos
limpios = []
for doc in crudos:
    texto_limpio = normalizar_texto(doc.page_content)
    if not texto_limpio:
        continue  # saltamos documentos sin contenido útil
    limpios.append(
        Document(
            page_content=texto_limpio,
            metadata=dict(doc.metadata),  # copia la metadata (source, distrito, etc.)
        )
    )

# Paso 3: trocear documentos largos en chunks más pequeños
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
chunks = splitter.split_documents(limpios)

# Paso 4: numerar cada chunk (chunk_index servirá al indexar en Chroma)
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_index"] = i

print(f"Documentos cargados: {len(crudos)}")
print(f"Documentos tras limpieza: {len(limpios)}")
print(f"Chunks generados: {len(chunks)}")

  206974-3-agenda-eventos-culturales-100.pdf: +3 documento(s)
  206974-4-agenda-eventos-culturales-100-csv.csv: +757 documento(s)
  faq_agenda_cultural.md: +1 documento(s)
  guia_agenda_cultural.txt: +1 documento(s)
  README.md: +1 documento(s)
Documentos cargados: 763
Documentos tras limpieza: 763
Chunks generados: 1334


## Recap Sprint 8 — Embeddings con Gemini

Cada chunk de texto se convierte en un **vector de números** que representa su significado.

- Usamos el modelo **`gemini-embedding-2`** (el mismo que en Sprint 8).
- **`MAX_CHUNKS_EMBED`** limita cuántos chunks embeddeamos en esta demo (ahorra tiempo y coste de API).
- Guardamos el resultado en `embeddings.json` para poder inspeccionarlo o reindexar sin volver a llamar a la API.

Cada item del JSON tiene: `text`, `vector` y `metadata`.

In [9]:
def embeddear_textos(textos: list[str]) -> list[list[float]]:
    """Pide a Gemini un vector por cada texto. Devuelve lista de listas de floats."""
    if not textos:
        return []

    # Un types.Content por texto → un embedding independiente por chunk
    contents = []
    for t in textos:
        contents.append(types.Content(parts=[types.Part(text=t)]))

    result = client.models.embed_content(model=EMBEDDING_MODEL, contents=contents)

    vectores = []
    for embedding in result.embeddings:
        vectores.append(list(embedding.values))

    return vectores


# Paso 1: elegir cuántos chunks embeddear (demo rápida vs corpus completo)
if MAX_CHUNKS_EMBED is None:
    subset = chunks
else:
    subset = chunks[:MAX_CHUNKS_EMBED]

# Paso 2: extraer solo el texto de cada chunk
textos = []
for chunk in subset:
    textos.append(chunk.page_content)

# Paso 3: llamar a la API y obtener vectores
vectores = embeddear_textos(textos)

# Paso 4: unir cada chunk con su vector correspondiente (mismo orden)
items = []
for chunk, vector in zip(subset, vectores):
    items.append({
        "text": chunk.page_content,
        "vector": vector,
        "metadata": dict(chunk.metadata),
    })

# Paso 5: guardar en JSON para inspeccionar o reindexar sin repetir la API
payload = {
    "embedding_model": EMBEDDING_MODEL,
    "total": len(items),
    "total_chunks_en_origen": len(chunks),
    "dimensions": len(vectores[0]) if vectores else 0,
    "items": items,
}
EMBEDDINGS_JSON.write_text(
    json.dumps(payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Guardado en output/embeddings.json")
print(f"  Vectores embeddeados: {len(items)} (de {len(chunks)} chunks totales)")
print(f"  Dimensiones por vector: {payload['dimensions']}")

Guardado en output/embeddings.json
  Vectores embeddeados: 20 (de 1334 chunks totales)
  Dimensiones por vector: 3072


## Paso clave del sprint 9 — Crear cliente y colección ChromaDB

Hasta aquí teníamos vectores en un JSON. **ChromaDB** los organiza para búsqueda rápida por similitud.

- **`PersistentClient`** guarda el índice en disco (`chroma_db/`), no se pierde al cerrar el notebook.
- Una **colección** es como una tabla de vectores de un mismo corpus.
- `hnsw:space: cosine` indica que mediremos similitud con **coseno** (habitual en embeddings).

Borramos la colección si ya existía para evitar IDs duplicados al re-ejecutar el notebook.

In [10]:
# Crear carpeta de persistencia si no existe
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

# Cliente persistente: guarda el índice en disco (no se pierde al cerrar el notebook)
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Si re-ejecutas el notebook, borramos la colección anterior para evitar IDs duplicados
try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print(f"Colección anterior '{COLLECTION_NAME}' eliminada.")
except Exception:
    print(f"No había colección previa llamada '{COLLECTION_NAME}'.")

# Crear (o recuperar) la colección donde guardaremos los vectores
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},  # mediremos similitud con coseno
)

print("Colección lista:", collection.name)

No había colección previa llamada 'agenda_cultural_madrid'.
Colección lista: agenda_cultural_madrid


## Indexar: de embeddings.json a ChromaDB

La operación central es **`collection.add()`**. Por cada chunk enviamos cuatro cosas:

| Campo | Qué es |
|-------|--------|
| `ids` | Identificador único (`chunk_0`, `chunk_1`, …) |
| `embeddings` | El vector numérico |
| `documents` | El texto del fragmento |
| `metadatas` | De dónde vino (`source`, `distrito`, etc.) |

Chroma solo acepta metadatos simples (texto, números, booleanos), por eso **sanitizamos** la metadata antes de indexar.

In [11]:
def sanitizar_metadata(metadata: dict) -> dict:
    """Chroma solo acepta metadatos simples: str, int, float o bool."""
    limpia = {}
    for clave, valor in metadata.items():
        if valor is None:
            continue  # omitimos valores vacíos
        if isinstance(valor, (str, int, float, bool)):
            limpia[clave] = valor
        else:
            limpia[clave] = str(valor)  # convertimos otros tipos a texto
    return limpia


# Preparamos cuatro listas paralelas (mismo índice = mismo chunk)
ids = []
embeddings = []
documents = []
metadatas = []

for i, item in enumerate(items):
    meta = sanitizar_metadata(item["metadata"])

    # ID único por chunk (usamos chunk_index si existe)
    chunk_index = meta.get("chunk_index", i)
    ids.append(f"chunk_{chunk_index}")

    embeddings.append(item["vector"])   # el vector numérico
    documents.append(item["text"])      # el texto del fragmento
    metadatas.append(meta)              # de dónde vino (source, distrito, etc.)

# Indexar todo de una vez en ChromaDB
collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=documents,
    metadatas=metadatas,
)

print(f"Total indexado en '{COLLECTION_NAME}':", collection.count())

Total indexado en 'agenda_cultural_madrid': 20


## Comprobar que el índice tiene sentido

Antes de nada, **vamos a inspeccionar qué hay dentro** de la colección.

`peek(limit=2)` muestra una muestra de chunks indexados: texto y metadata. 

Pregúntate:
- ¿El número de vectores coincide con lo que embeddeaste?
- ¿La metadata tiene `source` para saber de qué archivo vino cada fragmento?

In [12]:
peek = collection.peek(limit=2)
for i, doc_id in enumerate(peek["ids"]):
    print(f"--- {doc_id} ---")
    print(peek["documents"][i][:200], "...")
    print("metadata:", peek["metadatas"][i])

--- chunk_0 ---
Estructura para Eventos provenientes de www.madrid.es


La información existente en este conjunto de datos, proviene de la página web municipal www.madrid.es.
Esta estructura de información, es genéri ...
metadata: {'moddate': '2020-02-25T09:27:33+01:00', 'chunk_index': 0, 'page_label': '1', 'creationdate': '2020-02-25T09:27:33+01:00', 'page': 0, 'creator': 'Microsoft® Word 2013', 'total_pages': 3, 'producer': 'Microsoft® Word 2013', 'source': 'C:\\Users\\xandr\\Documents\\TheBridge\\bootcamp_ai_engineering\\material_AI_Engineering\\05_RAG_Engineering\\Sprint_09\\02_Workout\\data\\206974-3-agenda-eventos-culturales-100.pdf'}
--- chunk_1 ---
El concepto tiempo real asociado a e stos conjuntos de datos , significa que se está dando la última
información disponible en la Web municipal: la misma se actualiza en cuanto se conoce un cambio en  ...
metadata: {'total_pages': 3, 'source': 'C:\\Users\\xandr\\Documents\\TheBridge\\bootcamp_ai_engineering\\material_AI_Engineering\\

## Cierre y siguientes pasos

Has completado la fase de **indexación**:

```text
documentos → chunks → embeddings → ChromaDB
```

**Archivos generados** (en `02_Workout/output/`):
- `embeddings.json` — copia legible de los vectores.
- `chroma_db/` — índice listo para búsqueda.

El **siguiente paso será** hacer una pregunta al sistema y recuperar los chunks más similares.